# EDA — UCI Credit Card et Bank Marketing

Les trois tables sont analysées séparément : Credit Card Clients, Bank Marketing `bank-full` et Bank Marketing Additional `bank-additional-full`. Elles proviennent de populations, de périodes et d'objectifs distincts ; elles ne sont jamais fusionnées.

Les contrôles couvrent les doublons, les valeurs manquantes et codées `unknown`, les codes catégoriels hors nomenclature, les distributions et les associations avec chaque cible. Les tests sont exploratoires et ne démontrent pas de causalité.

**Alerte de fuite :** dans les deux jeux Bank Marketing, `duration` est la durée du dernier appel. La documentation source précise qu'elle n'est pas connue avant cet appel et qu'elle influence fortement la cible. Elle est montrée pour l'EDA, mais doit être retirée d'un modèle de décision avant contact client.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import chi2_contingency, mannwhitneyu

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "data" / "raw").exists():
    ROOT_DIR = ROOT_DIR.parent
RAW_DIR = ROOT_DIR / "data" / "raw"
sns.set_theme(style="whitegrid")
print(f"Données brutes : {RAW_DIR}")

## Fonctions de contrôle

Les tests de Mann–Whitney comparent les distributions de chaque variable numérique entre les deux classes. Les tailles d'effet rang-bisériales sont rapportées avec les médianes ; les p-values seules ne mesurent pas l'importance métier.

In [ ]:
def quality_summary(frame, target, name):
    """Afficher les dimensions, doublons, manquants et la distribution cible."""
    print(f"\n## {name}")
    print(f"Forme : {frame.shape[0]:,} lignes × {frame.shape[1]} colonnes")
    print(f"Doublons de lignes complets : {frame.duplicated().sum():,}")
    print(f"Valeurs manquantes explicites : {frame.isna().sum().sum():,}")
    display(
        frame[target].value_counts(dropna=False)
        .rename("effectif")
        .to_frame()
        .assign(proportion=lambda table: table["effectif"] / len(frame))
    )
    missing_report = pd.DataFrame(
        {
            "manquants": frame.isna().sum(),
            "manquants_pct": frame.isna().mean() * 100,
        }
    ).query("manquants > 0")
    if missing_report.empty:
        print("Aucune valeur NaN explicite.")
    else:
        display(missing_report.sort_values("manquants_pct", ascending=False).round(2))


def mann_whitney_report(frame, target, positive_label, columns):
    """Tester une famille définie de colonnes numériques et corriger les p-values."""
    results = []
    positive = frame[target].eq(positive_label)
    for column in columns:
        positive_values = frame.loc[positive, column].dropna()
        negative_values = frame.loc[~positive, column].dropna()
        test = mannwhitneyu(positive_values, negative_values, alternative="two-sided", method="asymptotic")
        p_value = float(test.pvalue)
        results.append(
            {
                "variable": column,
                "médiane_cible_positive": positive_values.median(),
                "médiane_cible_négative": negative_values.median(),
                "effet_rang_bisérial": 2 * test.statistic / (len(positive_values) * len(negative_values)) - 1,
                "p_brute": p_value,
                "p_bonferroni": min(p_value * len(columns), 1.0),
            }
        )
    return pd.DataFrame(results).set_index("variable")


def categorical_association(frame, target, columns):
    """Calculer le chi-deux d'indépendance pour les catégories choisies."""
    results = []
    for column in columns:
        table = pd.crosstab(frame[column].fillna("__NAN__"), frame[target])
        statistic, p_value, degrees_of_freedom, _ = chi2_contingency(table)
        results.append(
            {
                "variable": column,
                "chi2": statistic,
                "degrés_liberté": degrees_of_freedom,
                "p_brute": p_value,
                "p_bonferroni": min(float(p_value) * len(columns), 1.0),
            }
        )
    return pd.DataFrame(results).set_index("variable")

## 1. UCI Default of Credit Card Clients

La feuille Excel comporte une ligne de libellés supplémentaires : `header=1` donne les 25 colonnes de données attendues. La cible est `default payment next month`. Le champ `ID` est un identifiant, pas un prédicteur.

In [ ]:
uci_path = RAW_DIR / "uci_credit_card" / "default of credit card clients.xls"
if not uci_path.is_file():
    raise FileNotFoundError(f"Jeu UCI Credit Card absent : {uci_path}")

credit = pd.read_excel(uci_path, header=1, engine="calamine")
credit_target = "default payment next month"
quality_summary(credit, credit_target, "UCI Credit Card Clients")
print(f"ID unique : {credit['ID'].is_unique}; doublons de caractéristiques hors ID : {credit.drop(columns='ID').duplicated().sum():,}")
display(credit.describe(include="all").T.head(15))

In [ ]:
uci_expected_codes = {"SEX": {1, 2}, "EDUCATION": {1, 2, 3, 4}, "MARRIAGE": {1, 2, 3}}
code_quality = []
for column, allowed_codes in uci_expected_codes.items():
    invalid = ~credit[column].isin(allowed_codes)
    code_quality.append(
        {
            "variable": column,
            "codes_observés": sorted(credit[column].unique().tolist()),
            "hors_nomenclature": int(invalid.sum()),
            "hors_nomenclature_pct": invalid.mean() * 100,
        }
    )
display(pd.DataFrame(code_quality).set_index("variable"))
print("\nRetards PAY_0 :")
display(credit["PAY_0"].value_counts().sort_index().rename("effectif").to_frame())

for column in ["EDUCATION", "MARRIAGE", "PAY_0"]:
    rates = credit.groupby(column)[credit_target].agg(effectif="size", taux_defaut="mean")
    rates["taux_defaut_pct"] = rates["taux_defaut"] * 100
    print(f"\nTaux de défaut par {column} (descriptif) :")
    display(rates.drop(columns="taux_defaut").round(3))

uci_numeric_tests = mann_whitney_report(
    credit,
    credit_target,
    1,
    ["LIMIT_BAL", "AGE", "PAY_0", "PAY_2", "PAY_AMT1"],
)
print("\nHypothèses numériques (Mann–Whitney, correction Bonferroni × 5) :")
display(uci_numeric_tests.round(6))
uci_categorical_tests = categorical_association(
    credit, credit_target, ["SEX", "EDUCATION", "MARRIAGE", "PAY_0", "PAY_2"]
)
print("Tests du chi-deux d'indépendance (correction Bonferroni × 5) :")
display(uci_categorical_tests.round(6))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.histplot(data=credit, x="LIMIT_BAL", hue=credit_target, bins=35, stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Limite de crédit")
sns.countplot(data=credit, x="PAY_0", hue=credit_target, ax=axes[1])
axes[1].set_title("Statut de remboursement PAY_0")
limit_groups = pd.qcut(credit["LIMIT_BAL"], q=10, duplicates="drop")
limit_default_rate = credit.groupby(limit_groups, observed=True)[credit_target].mean().mul(100)
limit_default_rate.plot.bar(ax=axes[2], color="darkorange")
axes[2].set_title("Défaut par décile de limite")
axes[2].set_ylabel("Taux de défaut (%)")
axes[2].tick_params(axis="x", labelrotation=60)
plt.tight_layout()
plt.show()

## 2. Bank Marketing — deux sources séparées

`bank-full.csv` et `bank-additional-full.csv` décrivent deux campagnes et jeux de variables différents : leurs lignes ne doivent pas être concaténées. Le code `unknown` est une catégorie fournie par la source, pas un `NaN` pandas ; on le quantifie sans le remplacer.

Le code `pdays=-1` dans `bank-full` et `pdays=999` dans `bank-additional-full` signifie que le client n'a pas été contacté lors d'une campagne précédente. Ce code doit être distingué d'un nombre de jours réel.

La documentation UCI signale que `duration` est inconnue avant le dernier appel et qu'après l'appel le résultat de souscription est déjà connu. Les graphiques l'incluent comme variable descriptive, mais un modèle prospectif de ciblage client doit exclure cette colonne.

In [ ]:
bank_specs = [
    {
        "name": "Bank Marketing bank-full",
        "path": RAW_DIR / "bank_marketing" / "bank-full.csv",
        "sentinel": -1,
        "numeric_tests": ["age", "balance", "campaign", "previous"],
    },
    {
        "name": "Bank Marketing Additional bank-additional-full",
        "path": RAW_DIR / "bank_marketing" / "bank-additional-full.csv",
        "sentinel": 999,
        "numeric_tests": ["age", "campaign", "previous", "emp.var.rate", "euribor3m"],
    },
]
bank_frames = {}
bank_reports = {}
for spec in bank_specs:
    if not spec["path"].is_file():
        raise FileNotFoundError(f"Jeu Bank Marketing absent : {spec['path']}")
    frame = pd.read_csv(spec["path"], sep=";", low_memory=False)
    bank_frames[spec["name"]] = frame
    quality_summary(frame, "y", spec["name"])
    print(f"\nValeurs codées unknown par variable :")
    unknown_counts = frame.select_dtypes(include="object").eq("unknown").sum()
    display(unknown_counts.loc[unknown_counts.gt(0)].rename("unknown_effectif").to_frame())
    sentinel_count = int(frame["pdays"].eq(spec["sentinel"]).sum())
    print(f"pdays sentinelle ({spec['sentinel']}) : {sentinel_count:,} ({sentinel_count / len(frame):.2%})")
    print(f"Durée médiane du dernier appel, oui/non : {frame.loc[frame['y'].eq('yes'), 'duration'].median():.1f} / {frame.loc[frame['y'].eq('no'), 'duration'].median():.1f} secondes")
    display(frame.describe(include="all").T.head(12))

    bank_reports[spec["name"]] = {
        "numeric": mann_whitney_report(frame, "y", "yes", spec["numeric_tests"]),
        "categorical": categorical_association(
            frame, "y", ["job", "education", "contact", "poutcome", "month"]
        ),
    }
    print("\nTests numériques exploratoires (Mann–Whitney, Bonferroni) :")
    display(bank_reports[spec["name"]]["numeric"].round(6))
    print("Tests catégoriels (chi-deux, Bonferroni) :")
    display(bank_reports[spec["name"]]["categorical"].round(6))
    print("Taux de souscription selon le résultat de la campagne précédente :")
    previous_outcome = frame.groupby("poutcome")["y"].apply(lambda values: values.eq("yes").mean() * 100)
    display(previous_outcome.rename("souscription_pct").round(2).to_frame())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, spec in zip(axes.flat, bank_specs):
    frame = bank_frames[spec["name"]]
    sns.histplot(data=frame, x="duration", hue="y", bins=45, stat="density", common_norm=False, ax=ax)
    ax.set_title(f"{spec['name']} — durée (descriptive, fuite pour prédiction avant appel)")

for ax, spec in zip(axes.flat[2:], bank_specs):
    frame = bank_frames[spec["name"]]
    rates = frame.groupby("poutcome")["y"].apply(lambda values: values.eq("yes").mean() * 100).sort_values()
    rates.plot.barh(ax=ax, color="teal")
    ax.set_title(f"{spec['name']} — réponse par résultat antérieur")
    ax.set_xlabel("Souscription observée (%)")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()

## Synthèse et décisions de préparation

- **UCI Credit Card :** préserver la cible déséquilibrée dans toutes les analyses ; traiter `PAY_0` à `PAY_6` comme des niveaux de statut plutôt que comme des montants continus. Les codes `EDUCATION=0/5/6` et `MARRIAGE=0` sont à documenter comme catégories hors nomenclature/indéterminées, sans les effacer.
- **Bank Marketing :** conserver `unknown` comme information explicite ou la traiter par une stratégie d'imputation apprise dans le pipeline ; distinguer la sentinelle `pdays` d'une vraie durée écoulée.
- **Prévision avant contact :** retirer `duration` ; rapporter séparément tout benchmark qui la conserverait. Pour `bank-full`, préserver l'ordre temporel documenté lors de la validation.
- **Qualité :** examiner les 12 lignes identiques de `bank-additional-full` avant toute déduplication et documenter toute décision. Les lignes UCI ayant des caractéristiques identiques hors ID ne sont pas automatiquement des doublons d'observation.
- Les taux par catégories et les tests sont descriptifs ; les grandes tailles rendent les p-values petites même pour des effets faibles. Ne pas en déduire une causalité ou une règle de décision individuelle.